In [ ]:
import pandas as pd
import torch
from transformers import GPT2Tokenizer, GPT2Model, AutoTokenizer, AutoModel

df = pd.read_csv("VLDDR_Centroid.csv", sep=";")

GPTtokenizer = GPT2Tokenizer.from_pretrained("gpt2")
GPTmodel = GPT2Model.from_pretrained("gpt2")
GPTmodel.eval()

E5tokenizer = AutoTokenizer.from_pretrained('intfloat/e5-small-v2')
E5model = AutoModel.from_pretrained('intfloat/e5-small-v2')
E5model.eval()

Calculating EOS for GPT\
This code ran in ~8 minutes on a Ryzen 5 2600 six core CPU

In [ ]:
def cosine_sim(v1, v2):
    norm_v1 = torch.nn.functional.normalize(v1, dim = 0)
    norm_v2 = torch.nn.functional.normalize(v2, dim = 0)
    return float(torch.dot(norm_v1, norm_v2))

def get_eos_embedding(sentence):
    # Append EOS token explicitly
    sentence = sentence + GPTtokenizer.eos_token
    encoded = GPTtokenizer(sentence, return_tensors="pt")
    with torch.no_grad():
        output = GPTmodel(**encoded)

    return output.last_hidden_state.squeeze(0)[-1]  # Last token = EOS


def eos_sim(sent1, sent2):
    emb1 = get_eos_embedding(sent1)
    emb2 = get_eos_embedding(sent2)
    return cosine_sim(emb1, emb2)

df["EOS_GPT"] = df.apply(
    lambda row: eos_sim(row["sentence1"], row["sentence2"]),
    axis=1
)

Calculating CLS for E5
This ran in ~2 minutes on a Ryzen 5 2600 6 core CPU

In [ ]:
def get_first_embedding(sentence):
    encoded = E5tokenizer(sentence, return_tensors="pt")
    with torch.no_grad():
        output = E5model(**encoded)

    return output.last_hidden_state.squeeze(0)[0] #

def eos_sim(sent1, sent2):
    contextual1 = get_first_embedding(sent1)
    contextual2 = get_first_embedding(sent2)
    return cosine_sim(contextual1, contextual2)

df["CLS_E5"] = df.apply(
    lambda row: eos_sim(row["sentence1"], row["sentence2"]),
    axis=1
)

In [ ]:
df = df.drop(columns= ["GPT_ID1", "GPT_ID2", "E5_ID1", "E5_ID2"])

In [ ]:
df.to_csv("FinalData.csv", sep=";", index=False)